# 🟩 Pattern 2 — Reasoning Agents (ReAct)

> **One-line definition:** the agent interleaves *thinking* and *doing*, discovering the path as it executes.

ReAct = **Rea**son + **Act** (Yao et al., 2022).

---

## 1. Mental Model

```
        ┌───────────────────────────┐
        │                           │
        ▼                           │
     Reason  ──► Act  ──► Observe ──┘
   (LLM picks   (tool    (result goes
    an action)   runs)    back to LLM)
        │
        │ no more tool calls needed
        ▼
      Answer
```

The loop runs **until the LLM stops asking for tools**. Nobody knows in advance how many
iterations it will take — that's the whole point.

---

## 2. Key Properties (pointwise)

| Property | ReAct agent |
|---|---|
| Planning | ❌ no upfront plan |
| Loop | ✅ **cycle** (agent ⇄ tools) |
| Memory | 🟡 within-run only (the message list) |
| Reflection | ❌ no explicit critic |
| Termination | LLM decides (or a recursion cap) |
| Cost | 💰 variable, unbounded-ish |
| Predictability | 🟡 medium |

---

## 3. The confusing parts (resolved 👇)

### Q1: "Where is the 'Thought' step in the graph? I only see agent and tools."

**This is the #1 confusion.** In the *original paper*, ReAct was a **prompt format**:

```
Thought: I need the weather first
Action: get_weather[Cologne]
Observation: 12°C, rainy
Thought: Now I can answer
```

In **modern LangGraph ReAct**, the "Thought" is *not a separate node*. It's implicit in the
LLM's decision to emit a `tool_call` or not. The reasoning happens **inside** the `agent` node.

👉 **Thought = the LLM call. Action = the tool_call it emits. Observation = the ToolMessage.**

The graph only needs 2 nodes because thinking and choosing are the same LLM call.

---

### Q2: "How does the loop actually terminate?"

Purely by the shape of the LLM's response:

| LLM returns | Router sends to | Meaning |
|---|---|---|
| `AIMessage` **with** `.tool_calls` | `tools` node | "I need more info" |
| `AIMessage` **without** `.tool_calls` | `END` | "I'm done, here's the answer" |

There is **no counter, no plan, no goal check**. That's it. If the LLM never stops,
LangGraph's `recursion_limit` (default 25) raises an error — your only real safety net.

---

### Q3: "Is ReAct the same as tool calling?"

No — tool calling is the *mechanism*; ReAct is the *loop around it*.

- **Reactive + tools** = call tool once, format, stop. (Pattern 1)
- **ReAct** = call tool, **feed result back**, let LLM decide again. Repeat.

The back-edge `tools → agent` is the entire difference.

---

### Q4: "Why does the tool result have to go back as a `ToolMessage`?"

Because the LLM APIs require it. Every `tool_call` **must** be answered by a `ToolMessage`
carrying the same `tool_call_id`. If you skip one, the provider throws a 400.
`ToolNode` handles this bookkeeping for you — that's why you should use it.

---

## 4. Graph we will build

```
        START
          │
          ▼
      ┌───────┐
   ┌─►│ agent │  (LLM: reason + choose)
   │  └───────┘
   │      │ conditional edge
   │      ├──── has tool_calls? ──► ┌───────┐
   │      │                         │ tools │
   │      │                         └───────┘
   │      │                             │
   └──────┴─────────────────────────────┘   ◄── THE CYCLE
          │
          └──── no tool_calls ──────► END
```


## 0. Setup

**Install once:**

```bash
pip install langgraph langchain-openai langchain-core
```

**Set your API key** (any chat model works — swap the import if you use Anthropic/Ollama).


In [ ]:
# --- Standard setup used by every notebook in this series ---
import os, getpass

def _set(var: str):
    """Prompt for a key only if it's not already in the environment."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set("GROQ_API_KEY")

from langchain_groq import ChatGroq

# temperature=0 -> deterministic-ish output, easier to reason about while learning
llm = ChatGroq(model="qwen/qwen3-32b", temperature=0.4, reasoning_format="hidden")
print("LLM ready")

---

## 5. Step 1 — Tools

**Tool design rules (these matter more than your prompt):**

1. The **docstring is the spec the LLM reads** — write it for the model, not for humans.
2. **Type hints become the JSON schema.** Untyped args = broken tools.
3. Keep each tool **narrow**. One tool that does 5 things confuses the model.
4. Return **strings or simple values** — they get serialised into the message list.
5. Handle errors *inside* the tool and return a helpful message, so the LLM can self-correct.


In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use for any weather-related question."""
    data = {
        "cologne": "12°C, light rain",
        "vienna":  "18°C, sunny",
        "salzburg": "9°C, overcast",
    }
    # Return a friendly miss instead of raising -> lets the LLM recover on its own.
    return data.get(city.lower(), f"No weather data for {city}")


@tool
def get_activities(city: str, weather_condition: str) -> str:
    """Suggest activities for a city given the weather condition
    (e.g. 'rain', 'sunny', 'snow'). Call get_weather FIRST to learn the condition."""
    if "rain" in weather_condition.lower():
        return f"Indoor picks in {city}: museums, thermal baths, cafés"
    return f"Outdoor picks in {city}: parks, walking tours, river promenade"


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression, e.g. '23 * 4 + 10'."""
    try:
        # eval is fine for a demo; in production use a real parser (e.g. numexpr/asteval)
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"       # LLM sees this and can retry with a fixed expression


tools = [get_weather, get_activities, calculator]

# bind_tools() attaches the JSON schemas to the model so it CAN emit tool_calls.
# It does NOT execute anything — execution is our job (or ToolNode's).
llm_with_tools = llm.bind_tools(tools)
print("Tools bound:", [t.name for t in tools])

### 👀 See what a tool call actually looks like

This makes the whole pattern click — the LLM doesn't *run* anything. It just *asks*.


In [ ]:
resp = llm_with_tools.invoke("What's the weather in Cologne?")
print("content :", repr(resp.content))       # usually empty when a tool is requested
print("tool_calls:", resp.tool_calls)        # <- the "Action" from the ReAct paper

---

## 6. Step 2 — State

`MessagesState` is a prebuilt shortcut for:

```python
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

**Why the `add_messages` reducer matters:**

| Without reducer | With `add_messages` |
|---|---|
| new list **replaces** old | new messages **append** |
| conversation is destroyed each node | history accumulates |
| ReAct is impossible | ReAct works |

`add_messages` also de-duplicates by message `id` and lets you *overwrite* a message by
reusing its id — handy for trimming.


In [ ]:
from langgraph.graph import MessagesState

# Optional: extend it if you need extra fields
class ReActState(MessagesState):
    """MessagesState already gives us: messages (with add_messages reducer).
    Subclass it to add your own keys."""
    step_count: int

---

## 7. Step 3 — The two nodes

### Node A: `agent` — this IS the "Reason" step


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM = SystemMessage(content=(
    "You are a helpful travel assistant. "
    "Use tools when you need facts. Think step by step: if one tool's output is "
    "needed as another tool's input, call them in sequence."
))


def agent_node(state: ReActState) -> dict:
    """The LLM reasons over the FULL history and either:
       (a) emits tool_calls  -> loop continues
       (b) emits plain text  -> loop ends
    """
    # Pass the whole message list -> the LLM sees every previous observation.
    # This accumulated history IS the agent's working memory.
    response = llm_with_tools.invoke([SYSTEM] + state["messages"])

    # add_messages reducer appends this; it does not overwrite.
    return {"messages": [response], "step_count": state.get("step_count", 0) + 1}

### Node B: `tools` — the "Act" + "Observe" step

`ToolNode` (prebuilt) does all of this for you:
1. Reads `tool_calls` from the last `AIMessage`
2. Runs **all** of them (in parallel if there are several)
3. Wraps each result in a `ToolMessage` with the matching `tool_call_id`
4. Catches exceptions and returns them as `ToolMessage` content so the LLM can recover


In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

# --- The manual equivalent, so you know what ToolNode is doing ---
# from langchain_core.messages import ToolMessage
# def tool_node_manual(state):
#     out = []
#     for call in state["messages"][-1].tool_calls:
#         result = {t.name: t for t in tools}[call["name"]].invoke(call["args"])
#         out.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
#     return {"messages": out}   # <- the id MUST match, or the API errors

---

## 8. Step 4 — The router

This one function is the heart of ReAct.


In [ ]:
from langgraph.graph import END

def should_continue(state: ReActState) -> str:
    """Look at the last message. Tool calls -> keep looping. Otherwise -> stop."""
    last = state["messages"][-1]

    # `getattr` guard: only AIMessage has .tool_calls
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

# NOTE: LangGraph also ships this exact logic as `tools_condition`:
#   from langgraph.prebuilt import tools_condition
# It returns "tools" or END. We wrote it by hand so the mechanism is visible.

---

## 9. Step 5 — Build the graph (note the back-edge!)


In [ ]:
from langgraph.graph import StateGraph, START

builder = StateGraph(ReActState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

# Conditional: agent -> tools  OR  agent -> END
builder.add_conditional_edges("agent", should_continue, ["tools", END])

# ⭐ THE BACK-EDGE. This single line turns a chain into a ReAct agent.
builder.add_edge("tools", "agent")

react_graph = builder.compile()
print(react_graph.get_graph().draw_mermaid())

In [ ]:
from IPython.display import Image, display
try:
    display(Image(react_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(rendering unavailable — see mermaid text above)")

---

## 10. Step 6 — Run it & watch the loop

This query **forces multi-step reasoning**: the agent cannot suggest activities until it
knows the weather. Watch it discover that dependency by itself.


In [ ]:
result = react_graph.invoke({
    "messages": [HumanMessage(content="What should I do in Cologne today, given the weather?")],
    "step_count": 0,
})

# Pretty-print the whole trajectory
for m in result["messages"]:
    m.pretty_print()

print(f"\nLLM calls made: {result['step_count']}")

### Streaming the loop step by step


In [ ]:
for chunk in react_graph.stream(
    {"messages": [HumanMessage(content="What is 23*4+10, and what's the weather in Vienna?")],
     "step_count": 0},
    stream_mode="updates",       # only show what each node returned
):
    for node, update in chunk.items():
        msg = update["messages"][-1]
        calls = getattr(msg, "tool_calls", None)
        print(f"[{node}] {calls if calls else msg.content[:120]}")

---

## 11. The 3-line version (`create_react_agent`)

Everything above is prebuilt. **Use this in real projects**; build it by hand only when you
need custom nodes.


In [ ]:
from langgraph.prebuilt import create_react_agent

quick_agent = create_react_agent(
    llm,
    tools=tools,
    prompt="You are a helpful travel assistant.",   # note: `prompt`, not `state_modifier`
)

out = quick_agent.invoke({"messages": [{"role": "user", "content": "Weather in Salzburg?"}]})
print(out["messages"][-1].content)

### Useful `create_react_agent` kwargs

| kwarg | Purpose |
|---|---|
| `prompt` | system prompt (str, SystemMessage, or callable) |
| `checkpointer` | adds cross-turn memory (see Pattern 5) |
| `store` | long-term memory across threads |
| `response_format` | force a structured final answer |
| `pre_model_hook` | trim/summarise messages before each LLM call |
| `post_model_hook` | guardrails / validation after each LLM call |
| `interrupt_before` | human-in-the-loop approval before tools run |


---

## 12. Safety: the recursion limit

**This is not optional in production.** A ReAct agent with a broken tool can loop forever.


In [ ]:
from langgraph.errors import GraphRecursionError

try:
    react_graph.invoke(
        {"messages": [HumanMessage(content="Weather in Cologne?")], "step_count": 0},
        config={"recursion_limit": 2},    # deliberately too low to show the guard firing
    )
except GraphRecursionError as e:
    print("Caught GraphRecursionError -> this is your production circuit breaker")
    print(e)

> 💡 **Rule of thumb:** `recursion_limit ≈ 2 × (max expected tool calls) + 1`.
> Each loop consumes 2 steps (agent + tools).

---

## 13. Cheat Sheet

```
DEFINITION   reason ⇄ act ⇄ observe, until the LLM stops asking for tools
GRAPH SHAPE  agent ⇄ tools  (a CYCLE)
THOUGHT      not a node — it's inside the agent's LLM call
TERMINATION  AIMessage without .tool_calls  -> END
MEMORY       the accumulating message list (within-run only)
GUARD        recursion_limit
USE WHEN     the path must be discovered from tool results
AVOID WHEN   steps are known (use Reactive) or plan must be reviewed (use Planning)
```

**API essentials**

| Task | Code |
|---|---|
| Define tool | `@tool` + typed args + docstring |
| Attach schemas | `llm.bind_tools(tools)` |
| Execute tools | `ToolNode(tools)` |
| Message state | `class S(MessagesState): ...` |
| Router | `tools_condition` or custom `-> "tools" \| END` |
| The cycle | `builder.add_edge("tools", "agent")` |
| Shortcut | `create_react_agent(llm, tools, prompt=...)` |
| Guard | `config={"recursion_limit": N}` |

---

## 14. Decision Tree

```
Does the agent need to discover its path from tool output?
├── NO ──────────────────────► Reactive (Pattern 1)
└── YES
    ├── Should the plan be visible/reviewable before acting?
    │   └── YES ─────────────► Planning (Pattern 3)
    ├── Must output quality be critiqued and revised?
    │   └── YES ─────────────► Reflective (Pattern 4)
    ├── Must it remember across sessions?
    │   └── YES ─────────────► ReAct + Memory (Pattern 5)
    └── Otherwise ───────────► ✅ ReAct (this notebook)
```

⚠️ **ReAct is the default choice for most agents.** Reach for the others only when you hit
a concrete limitation.

---

## 15. Next

➡️ **Pattern 3 — Planning Agents**: instead of discovering step-by-step, the agent writes
the whole plan first.
